# Import libraries

In [1]:
import FileReader as f
import Regression as r
import SQL as sql
import sqlalchemy as db
import Visualisation as visual
import Testing as test

In [3]:
# Finally, the training data, the test data, the chosen ideal functions as well as the corresponding /
# assigned datasets are visualized under an appropriately chosen representation of the deviation.

# It includes standard- und user-defined exception handlings
# Your code needs to be documented in its entirety and also include Documentation Strings, known as ”docstrings“


def main():

    df_train = f.FileReader.read_csv_to_df("data/train.csv")
    df_ideal = f.FileReader.read_csv_to_df("data/ideal.csv")

    regression = r.Regression(df_ideal)
    df_train_mse = df_train.apply(regression.mse)

    df_result = regression.min_value(df_train_mse, df_train)

    
    pu = sql.PersistenceUtil()
    
    pu.save_df_to_db(df_train, "train", r"Y\1 (training func)")
    pu.save_df_to_db(df_ideal, "ideal", r"Y\1 (ideal func)")
    
    chosen_ideal = df_ideal[df_result.min_y.tolist()]
    #print(chosen_ideal)

    data = regression.match("data/test.csv", df_result, chosen_ideal)

    test_table = pu.create_table("test", {"X (test func)": db.Float, "Y (test func)": db.Float, \
                         "Delta Y (test func)": db.Float, "No. of ideal func": db.String(5)}, drop_if_exists=True)
    
    try:
        pu.persist(test_table, data)
    except Exception as e:
        print("Could not persist data, continue anyway: {}".format(e.__str__()))
    
    visual.PlotFactory.show("Training Data", df_train, df_result, chosen_ideal)
    visual.PlotFactory.show("Test Data", data, df_result, chosen_ideal, vtype="test")

    print("Done")

if __name__ == '__main__':
    main()

Done
